# Foundation Brain — Colab T4 Extras

Runs the experiments that benefit from a faster/different GPU than the
project's local Apple Silicon (MPS) machine:

1. **No-pretraining control** — random-init backbone + linear probe. The key
   control for the paper's central claim: if this scores near chance while
   the NT-Xent-pretrained backbone scores meaningfully above it, that
   substantiates the pretraining is doing real work.
2. **Larger-batch-size pretraining** — NT-Xent quality scales with batch
   size (more in-batch negatives per anchor); MPS-constrained batch sizes
   were small (64), a T4 has headroom for 128–256.
3. **Longer pretraining schedule** — more epochs on the same 23-subject VF
   dataset, to check whether modest cross-task transfer is a pretraining-
   budget limitation rather than an architecture limitation.

**Runtime:** Set Runtime → Change runtime type → T4 GPU before running.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


name, memory.total [MiB]
Tesla T4, 15360 MiB


## 1. Clone repo and install dependencies

In [ ]:
!git clone https://github.com/aharshit123456/foundation-brain.git
%cd foundation-brain
!pip install -q numpy scipy torch scikit-learn tqdm


Cloning into 'foundation-brain'...
remote: Enumerating objects: 141, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 141 (delta 51), reused 127 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (141/141), 2.58 MiB | 35.73 MiB/s, done.
Resolving deltas: 100% (51/51), done.
/content/foundation-brain


## 2. Download dataset (all 23 subjects: VP001-011, VP014-025)

This re-downloads from the TU Berlin institutional repository — Colab's
disk is ephemeral, so we can't reuse the local Mac's already-downloaded
files. The institutional server is slow per-connection but not
meaningfully rate-limited per-IP (confirmed locally: 3 parallel batches
gave a ~3-4x speedup over one sequential call), so we split the subjects
into concurrent batches here too. Expect ~5-7 minutes instead of ~15-20.


In [ ]:
import subprocess

SUBJECTS = [f'VP{str(i).zfill(3)}' for i in range(1, 26) if i not in (12, 13)]
N_BATCHES = 4
batches = [SUBJECTS[i::N_BATCHES] for i in range(N_BATCHES)]

procs = []
for i, batch in enumerate(batches):
    log_path = f'/tmp/dl_colab_batch{i}.log'
    with open(log_path, 'w') as logf:
        p = subprocess.Popen(
            ['python3', 'data/download_dataset.py', *batch],
            stdout=logf, stderr=subprocess.STDOUT
        )
        procs.append((p, log_path, batch))
    print(f'Batch {i}: {batch}')

for p, log_path, batch in procs:
    p.wait()
    print(f'Batch {batch} finished (exit code {p.returncode})')


Batch 0: ['VP001', 'VP005', 'VP009', 'VP015', 'VP019', 'VP023']
Batch 1: ['VP002', 'VP006', 'VP010', 'VP016', 'VP020', 'VP024']
Batch 2: ['VP003', 'VP007', 'VP011', 'VP017', 'VP021', 'VP025']
Batch 3: ['VP004', 'VP008', 'VP014', 'VP018', 'VP022']
Batch ['VP001', 'VP005', 'VP009', 'VP015', 'VP019', 'VP023'] finished (exit code 0)
Batch ['VP002', 'VP006', 'VP010', 'VP016', 'VP020', 'VP024'] finished (exit code 0)
Batch ['VP003', 'VP007', 'VP011', 'VP017', 'VP021', 'VP025'] finished (exit code 0)
Batch ['VP004', 'VP008', 'VP014', 'VP018', 'VP022'] finished (exit code 0)


In [ ]:
# Sanity check: print the tail of each batch log to confirm no failures
for i in range(N_BATCHES):
    print(f'--- batch {i} ---')
    with open(f'/tmp/dl_colab_batch{i}.log') as f:
        print(f.read()[-500:])


--- batch 0 ---
19-NIRS.zip... done (29.2 MB)
  Extracting VP019-NIRS.zip... done
  Extracting VP019-EEG.zip... done

VP023:
  Extracting VP023-NIRS.zip... done
  Extracting VP023-EEG.zip... done

FAILED downloads: ['VP001-EEG']
Re-run the script to retry failed downloads.
Data saved to: /content/foundation-brain/data/raw

--- batch 1 ---
cting VP016-EEG.zip... done

VP020:
  Extracting VP020-NIRS.zip... done
  Extracting VP020-EEG.zip... done

VP024:
  Extracting VP024-NIRS.zip... done
  Extracting VP024-EEG.zip... done

All downloads complete.
Data saved to: /content/foundation-brain/data/raw

--- batch 2 ---
 done (29.2 MB)
  Extracting VP021-NIRS.zip... done
  Extracting VP021-EEG.zip... done

VP025:
  Extracting VP025-NIRS.zip... done
  Extracting VP025-EEG.zip... done

FAILED downloads: ['VP003-NIRS', 'VP011-EEG']
Re-run the script to retry failed downloads.
Data saved to: /content/foundation-brain/data/raw

--- batch 3 ---
8-NIRS.zip... done (29.2 MB)
  Extracting VP018-NIRS.zip

In [ ]:
!python3 preprocess_vf.py
!python3 preprocess_nback.py


Found available subjects for VF preprocessing: ['VP002', 'VP004', 'VP005', 'VP006', 'VP007', 'VP009', 'VP010', 'VP014', 'VP015', 'VP016', 'VP017', 'VP018', 'VP019', 'VP020', 'VP021', 'VP022', 'VP023', 'VP024', 'VP025']

--- Preprocessing VP002 ---
  EEG shape: (371416, 30), fs=200.0 Hz
  NIRS HbO shape: (20219, 36), fs=10.0 Hz
  Label dist: VF=30, BL=30
  EEG windows shape: (60, 30, 2000)
  NIRS windows shape: (60, 72, 100)
  Labels dist: VF=30, BL=30

--- Preprocessing VP004 ---
  EEG shape: (371949, 30), fs=200.0 Hz
  NIRS HbO shape: (20115, 36), fs=10.0 Hz
  Label dist: VF=30, BL=30
  EEG windows shape: (60, 30, 2000)
  NIRS windows shape: (60, 72, 100)
  Labels dist: VF=30, BL=30

--- Preprocessing VP005 ---
  EEG shape: (372647, 30), fs=200.0 Hz
  NIRS HbO shape: (20237, 36), fs=10.0 Hz
  Label dist: VF=30, BL=30
  EEG windows shape: (60, 30, 2000)
  NIRS windows shape: (60, 72, 100)
  Labels dist: VF=30, BL=30

--- Preprocessing VP006 ---
  EEG shape: (371874, 30), fs=200.0 Hz
  

## 3. No-Pretraining Control (the key addition)

Same LOSO protocol as `run_foundation_loso.py`, but the backbone is never
contrastively pretrained — random init, straight to the frozen linear probe.
If this scores near chance (~0.50 for the binary VF task) while the
pretrained backbone scores meaningfully higher, that's direct evidence the
NT-Xent objective is contributing real structure, not just providing a
fixed random projection that any logistic regression could exploit equally
well.


In [ ]:
!python3 run_foundation_nopretrain.py


Loading preprocessed VF windows...
Subjects: ['VP002', 'VP004', 'VP005', 'VP006', 'VP007', 'VP009', 'VP010', 'VP014', 'VP015', 'VP016', 'VP017', 'VP018', 'VP019', 'VP020', 'VP021', 'VP022', 'VP023', 'VP024', 'VP025']
Using device: cuda
GPU: Tesla T4
LOSO folds (no pretrain):   0% 0/19 [00:00<?, ?fold/s, test=VP002]
  VP002: Acc=0.650  F1=0.649
LOSO folds (no pretrain):   5% 1/19 [00:02<00:46,  2.57s/fold, test=VP004]
  VP004: Acc=0.717  F1=0.715
LOSO folds (no pretrain):  11% 2/19 [00:03<00:27,  1.60s/fold, test=VP005]
  VP005: Acc=0.717  F1=0.717
LOSO folds (no pretrain):  16% 3/19 [00:04<00:20,  1.29s/fold, test=VP006]
  VP006: Acc=0.700  F1=0.700
LOSO folds (no pretrain):  21% 4/19 [00:05<00:17,  1.15s/fold, test=VP007]
  VP007: Acc=0.767  F1=0.766
LOSO folds (no pretrain):  26% 5/19 [00:06<00:14,  1.07s/fold, test=VP009]
  VP009: Acc=0.600  F1=0.596
LOSO folds (no pretrain):  32% 6/19 [00:07<00:13,  1.03s/fold, test=VP010]
  VP010: Acc=0.450  F1=0.423
LOSO folds (no pretrain):  37%

In [ ]:
import json
with open('data/processed/foundation_nopretrain_results.json') as f:
    nopretrain = json.load(f)
print(f"No-pretraining control: Acc={nopretrain['mean_acc']:.3f}  F1={nopretrain['mean_f1']:.3f}")
print(f"(Compare to chance=0.500 and the pretrained Foundation Brain result)")


No-pretraining control: Acc=0.625  F1=0.616
(Compare to chance=0.500 and the pretrained Foundation Brain result)


## 4. Larger Batch Size Pretraining (128 vs. the local default of 64)

NT-Xent's negative set is every other sample in the batch — bigger batches
give the contrastive loss more (and harder) negatives per anchor, typically
improving representation quality. This was VRAM-constrained on the local
Mac's MPS backend; a T4 (16GB) has room for this.


In [ ]:
import pickle
import sys
sys.path.insert(0, '.')
from run_foundation_loso import run_loso

with open('data/processed/dataset_vf_windows.pkl', 'rb') as f:
    all_data = pickle.load(f)

results_bs128, mean_acc_bs128, mean_f1_bs128 = run_loso(
    all_data, n_pretrain_epochs=50, batch_size=128, lr=3e-4, temperature=0.07
)
print(f"Batch size 128: Acc={mean_acc_bs128:.3f}  F1={mean_f1_bs128:.3f}")


Using device: cuda
GPU: Tesla T4
    pretraining: 100%|██████████| 50/50 [03:30<00:00,  4.38s/ep, loss=1.6136]
                                                                             
  VP002: Acc=0.667  F1=0.661  (pretrain loss=1.6136)
    pretraining: 100%|██████████| 50/50 [03:40<00:00,  4.41s/ep, loss=1.7483]
                                                                             
  VP004: Acc=0.583  F1=0.569  (pretrain loss=1.7483)
    pretraining: 100%|██████████| 50/50 [03:40<00:00,  4.41s/ep, loss=1.5329]
                                                                             
  VP005: Acc=0.650  F1=0.648  (pretrain loss=1.5329)
    pretraining: 100%|██████████| 50/50 [03:41<00:00,  4.43s/ep, loss=1.5065]
                                                                             
  VP006: Acc=0.767  F1=0.767  (pretrain loss=1.5065)
    pretraining: 100%|██████████| 50/50 [03:40<00:00,  4.41s/ep, loss=1.6639]
                                                     

In [ ]:
import json, os

output_bs128 = {
    'model': 'Foundation Brain (batch_size=128 ablation)',
    'dataset': 'Shin 2017 Dataset C (VF task)',
    'n_subjects': len(all_data),
    'evaluation': 'LOSO -- larger batch size ablation',
    'batch_size': 128,
    'n_pretrain_epochs': 50,
    'mean_acc': round(mean_acc_bs128, 4),
    'mean_f1': round(mean_f1_bs128, 4),
    'per_subject': {s: {'acc': round(v['acc'], 4), 'f1': round(v['f1'], 4)}
                    for s, v in results_bs128.items()}
}
os.makedirs('data/processed', exist_ok=True)
with open('data/processed/foundation_results_bs128.json', 'w') as f:
    json.dump(output_bs128, f, indent=2)
print('Saved -> data/processed/foundation_results_bs128.json')


Saved -> data/processed/foundation_results_bs128.json


## 5. Longer Pretraining Schedule (150 epochs vs. the local default of 50)

Tests whether the modest cross-task transfer result (n-back probe only
slightly above chance) is a pretraining-budget limitation -- contrastive
objectives are notoriously epoch-hungry -- rather than an architecture
limitation.


In [ ]:
results_ep150, mean_acc_ep150, mean_f1_ep150 = run_loso(
    all_data, n_pretrain_epochs=150, batch_size=64, lr=3e-4, temperature=0.07
)
print(f"150 epochs: Acc={mean_acc_ep150:.3f}  F1={mean_f1_ep150:.3f}")


Using device: cuda
GPU: Tesla T4
    pretraining: 100%|██████████| 150/150 [11:06<00:00,  4.43s/ep, loss=0.1560]
                                                                               
  VP002: Acc=0.750  F1=0.744  (pretrain loss=0.1560)
    pretraining: 100%|██████████| 150/150 [11:05<00:00,  4.43s/ep, loss=0.1549]
                                                                               
  VP004: Acc=0.600  F1=0.598  (pretrain loss=0.1549)
    pretraining: 100%|██████████| 150/150 [11:04<00:00,  4.42s/ep, loss=0.1813]
                                                                               
  VP005: Acc=0.667  F1=0.667  (pretrain loss=0.1813)
    pretraining: 100%|██████████| 150/150 [11:05<00:00,  4.44s/ep, loss=0.1510]
                                                                               
  VP006: Acc=0.650  F1=0.633  (pretrain loss=0.1510)
    pretraining: 100%|██████████| 150/150 [11:04<00:00,  4.43s/ep, loss=0.1546]
                                   

In [ ]:
output_ep150 = {
    'model': 'Foundation Brain (150-epoch pretraining ablation)',
    'dataset': 'Shin 2017 Dataset C (VF task)',
    'n_subjects': len(all_data),
    'evaluation': 'LOSO -- longer pretraining schedule ablation',
    'batch_size': 64,
    'n_pretrain_epochs': 150,
    'mean_acc': round(mean_acc_ep150, 4),
    'mean_f1': round(mean_f1_ep150, 4),
    'per_subject': {s: {'acc': round(v['acc'], 4), 'f1': round(v['f1'], 4)}
                    for s, v in results_ep150.items()}
}
with open('data/processed/foundation_results_ep150.json', 'w') as f:
    json.dump(output_ep150, f, indent=2)
print('Saved -> data/processed/foundation_results_ep150.json')


## 6. Download results to merge locally

Run this cell and download the zip, then unzip into `data/processed/` in
the local repo before re-running `generate_paper_tex.py`.


In [ ]:
!zip -j colab_extras_results.zip \
    data/processed/foundation_nopretrain_results.json \
    data/processed/foundation_results_bs128.json \
    data/processed/foundation_results_ep150.json

from google.colab import files
files.download('colab_extras_results.zip')
